# Function Calling Demo

This notebook demonstrates **manual JSON Schema function calling** using a simple simulated weather tool.

## Learning objectives

By the end of this workshop, you should understand:

1. How to create a normal Python function.
2. How to describe that function to an LLM using JSON Schema.
3. How the LLM selects a tool and generates arguments.
4. How the Python application executes the function.
5. How the tool result is returned to the LLM for the final answer.


## 1. Configure the OpenAI client

Create a `.env` file in the same folder as this notebook:

```text
OPENAI_API_KEY=your_api_key_here
```

Then run the cell below.


In [11]:
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()

# Change the model if needed.
MODEL = "gpt-5.6"

## 2. Create the actual Python function

For this workshop, we use **simulated weather data** so that the focus stays on function calling rather than external APIs.

At this stage, `get_weather()` is just a normal Python function. The LLM does not know it exists yet.


In [12]:
def get_weather(location):
    """
    Simulated weather function for workshop purposes.
    """

    weather_data = {
        "Singapore": {
            "temperature": 31,
            "condition": "Partly cloudy",
            "humidity": 75
        },
        "London": {
            "temperature": 18,
            "condition": "Rainy",
            "humidity": 80
        },
        "Tokyo": {
            "temperature": 27,
            "condition": "Sunny",
            "humidity": 65
        }
    }

    if location in weather_data:
        return {
            "location": location,
            **weather_data[location]
        }

    return {
        "location": location,
        "error": "Weather data is not available for this location."
    }


### Test the Python function directly

No LLM is involved in this test.


In [13]:
print(get_weather("Singapore"))

{'location': 'Singapore', 'temperature': 31, 'condition': 'Partly cloudy', 'humidity': 75}


## 3. Define the tool manually using JSON Schema

The JSON Schema tells the model:

- the tool name,
- what the tool does,
- which arguments it accepts,
- the type of each argument.

The schema does **not** contain the implementation of the Python function.


In [14]:
tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Get the current weather for a specified city.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city name, for example Singapore, London, or Tokyo."
                }
            },
            "required": ["location"],
            "additionalProperties": False
        },
        "strict": True
    }
]

## 4. Create the user's question


In [ ]:
user_question = "What's the weather like in Singapore?"

input_list = [
    {
        "role": "user",
        "content": user_question
    }
]

## 5. Agent Loop

In [22]:
while True:

    # Call the LLM
    response = client.responses.create(
        model=MODEL,
        input=input_list,
        tools=tools
    )

    # Preserve everything returned by the model
    input_list += response.output

    # Find function calls requested by the LLM
    function_calls = [
        item
        for item in response.output
        if item.type == "function_call"
    ]

    # -----------------------------------
    # No tool call -> final answer
    # -----------------------------------

    if not function_calls:
        print("\nFinal answer:")
        print(response.output_text)
        break

    # -----------------------------------
    # Execute requested tools
    # -----------------------------------

    for function_call in function_calls:

        print("\nLLM requested tool:")
        print(function_call.name)

        print("\nArguments:")
        print(function_call.arguments)

        # Convert JSON arguments to Python dictionary
        arguments = json.loads(function_call.arguments)

        # Execute the actual Python function
        if function_call.name == "get_weather":

            result = get_weather(
                location=arguments["location"]
            )

            print("\nTool result:")
            print(result)

            # Send tool result back to the LLM
            input_list.append(
                {
                    "type": "function_call_output",
                    "call_id": function_call.call_id,
                    "output": json.dumps(result)
                }
            )


LLM requested tool:
get_weather

Arguments:
{"location":"Shanghai"}

Tool result:
{'location': 'Shanghai', 'error': 'Weather data is not available for this location.'}

Final answer:
Sorry, I couldn’t retrieve current weather data for Shanghai. Please check a local weather service such as the China Meteorological Administration or your preferred weather app.


## 6. Exercises

Try modifying the notebook:

1. Ask for the weather in `London`.
2. Ask for the weather in `Tokyo`.
3. Ask for a city that is not in `weather_data`.

